# Set Cover: MomentumBuilder vs. Phased vs. Merged

This notebook evaluates three approaches on a hard 9-qubit Set Cover instance:

1. **`MomentumBuilder` (no SA)**: Grows the ansatz over N momentum layers with no further parameter optimisation.
2. **`momentum_sa_phased`**: Runs MomentumBuilder for N iterations, *then* runs Simulated Annealing on all parameters at once.
3. **`momentum_sa_merged`**: Interleaves growth and SA — after each momentum layer is added, SA runs immediately on the current parameter set.

Each approach is run for **10 trials**. Results are aggregated (mean ± std) and compared.

In [ ]:
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator

# --- Path Setup ---
vqa_root = os.path.abspath(os.path.join('../../../../'))
if vqa_root not in sys.path:
    sys.path.insert(0, vqa_root)

ansatz_pruning_dir = os.path.abspath(os.path.join('../../../'))
if ansatz_pruning_dir not in sys.path:
    sys.path.insert(0, ansatz_pruning_dir)

from AnsatzPruning import MomentumMonteCarlo, MomentumBuilder

print('Imports OK')

## 1. Set Cover Hamiltonian

A hard 9-qubit instance: universe `{1,2,3,4,5,6}` with 9 subsets.

In [ ]:
def get_subset_Hamiltonian(universe, subsets):
    num_subsets = len(subsets)
    total_op = SparsePauliOp(["I" * num_subsets], coeffs=[0.0])
    for element in universe:
        relevant_indices = [i for i, s in enumerate(subsets) if element in s]
        pauli_id = "I" * num_subsets
        total_op += SparsePauliOp([pauli_id], coeffs=[1.0])
        for idx in relevant_indices:
            z_str = ["I"] * num_subsets
            z_str[num_subsets - 1 - idx] = "Z"
            total_op += SparsePauliOp(["".join(z_str)], coeffs=[0.5])
            total_op += SparsePauliOp([pauli_id], coeffs=[-0.5])
        relevant_indices.sort()
        for i_idx in range(len(relevant_indices)):
            for j_idx in range(i_idx + 1, len(relevant_indices)):
                u, v = relevant_indices[i_idx], relevant_indices[j_idx]
                total_op += SparsePauliOp([pauli_id], coeffs=[0.5])
                z_u = ["I"] * num_subsets; z_u[num_subsets - 1 - u] = "Z"
                total_op += SparsePauliOp(["".join(z_u)], coeffs=[-0.5])
                z_v = ["I"] * num_subsets; z_v[num_subsets - 1 - v] = "Z"
                total_op += SparsePauliOp(["".join(z_v)], coeffs=[-0.5])
                z_uv = ["I"] * num_subsets
                z_uv[num_subsets - 1 - u] = "Z"; z_uv[num_subsets - 1 - v] = "Z"
                total_op += SparsePauliOp(["".join(z_uv)], coeffs=[0.5])
    return total_op.simplify()


universe = [1, 2, 3, 4, 5, 6]
subsets = [
    {1, 2}, {3, 4}, {5, 6}, {1, 3, 5},
    {2, 4, 6}, {1, 4}, {2, 5}, {3, 6}, {1, 2, 6}
]

H = get_subset_Hamiltonian(universe, subsets)
num_qubits = len(subsets)
matrix = H.to_matrix()
diagonal = np.real(np.diag(matrix))
min_energy = np.min(diagonal)
valid_solutions = np.where(np.isclose(diagonal, min_energy))[0]

print(f'Hamiltonian on {num_qubits} qubits')
print(f'Ground state energy (classical): {min_energy:.6f}')
print(f'Number of valid solutions: {len(valid_solutions)}')
for sol in valid_solutions:
    bs = bin(sol)[2:].zfill(num_qubits)
    chosen = [i for i in range(num_qubits) if bs[num_qubits - 1 - i] == '1']
    print(f'  |{bs}> -> Subsets: {chosen}')

## 2. Helpers

In [ ]:
estimator = Estimator()
simulator = AerSimulator(method='statevector')

def make_initial_ansatz(num_qubits):
    params_symbols = [Parameter(f'a{i}') for i in range(num_qubits)]
    circuit = QuantumCircuit(num_qubits)
    ansatz = QuantumCircuit(num_qubits)
    for i in range(num_qubits):
        ansatz.rx(params_symbols[i], i)
    return circuit, ansatz, [1.0] * num_qubits, list(range(num_qubits))

def sv_energy(circ, params, H):
    """Bind params, run statevector, return expectation value."""
    bound = circ.assign_parameters(params)
    bound.save_statevector()
    sv = simulator.run(transpile(bound, simulator)).result().get_statevector()
    return float(np.real(sv.expectation_value(H)))

print('Helpers ready.')

## 3. Run 10 Trials for Each Approach

Each trial re-initialises the ansatz from scratch so there is no state carry-over between trials.

In [ ]:
NUM_TRIALS = 10
ITERS = 3
OPT_RUNS = 200

phased_energies, phased_times = [], []
merged_energies, merged_times = [], []
mb_energies,     mb_times     = [], []

# MomentumBuilder expects the full observables list (Pauli terms + Hamiltonian)
observables_list = [*H.paulis, H]

for trial in range(1, NUM_TRIALS + 1):
    print(f'Trial {trial}/{NUM_TRIALS}', end=' ... ')

    # --- Phased ---
    circuit_p, ansatz_p, params_p, inds_p = make_initial_ansatz(num_qubits)
    t0 = time.perf_counter()
    phased_circuit, phased_params, e_phased = MomentumMonteCarlo.momentum_sa_phased(
        params_p, inds_p, ansatz_p, circuit_p, H, estimator,
        beta1=0.9, beta2=0.99, iters=ITERS, optimization_runs=OPT_RUNS
    )
    t_phased = time.perf_counter() - t0
    phased_energies.append(e_phased)
    phased_times.append(t_phased)

    # --- Merged ---
    circuit_m, ansatz_m, params_m, inds_m = make_initial_ansatz(num_qubits)
    t0 = time.perf_counter()
    merged_circuit, merged_params, e_merged = MomentumMonteCarlo.momentum_sa_merged(
        params_m, inds_m, ansatz_m, circuit_m, H, estimator,
        beta1=0.9, beta2=0.99, iters=ITERS, optimization_runs=OPT_RUNS
    )
    t_merged = time.perf_counter() - t0
    merged_energies.append(e_merged)
    merged_times.append(t_merged)

    # --- Plain MomentumBuilder (no SA) ---
    circuit_b, ansatz_b, params_b, inds_b = make_initial_ansatz(num_qubits)
    t0 = time.perf_counter()
    mb_circuit = MomentumBuilder.MomentumBuilder(
        params_b, inds_b, ansatz_b, circuit_b, observables_list, estimator,
        0.9, 0.99, iters=ITERS
    )
    t_mb = time.perf_counter() - t0
    # MomentumBuilder returns an unbound circuit — evaluate at all-ones params
    mb_num_params = len(mb_circuit.parameters)
    e_mb = sv_energy(mb_circuit, np.ones(mb_num_params), H)
    mb_energies.append(e_mb)
    mb_times.append(t_mb)

    print(f'Phased={e_phased:.4f} ({t_phased:.1f}s)  '
          f'Merged={e_merged:.4f} ({t_merged:.1f}s)  '
          f'MB={e_mb:.4f} ({t_mb:.1f}s)')

print('\nAll trials complete.')

## 4. Comparison Summary & Charts

In [ ]:
phased_energies = np.array(phased_energies)
merged_energies = np.array(merged_energies)
mb_energies     = np.array(mb_energies)
phased_times    = np.array(phased_times)
merged_times    = np.array(merged_times)
mb_times        = np.array(mb_times)

print('=' * 70)
print(f'{"Method":<28} {"Mean Energy":>12} {"Std Energy":>11} {"Mean Time":>10}')
print('-' * 63)
print(f'{"MomentumBuilder (no SA)":<28} {mb_energies.mean():>12.4f} {mb_energies.std():>11.4f} {mb_times.mean():>9.2f}s')
print(f'{"momentum_sa_phased":<28} {phased_energies.mean():>12.4f} {phased_energies.std():>11.4f} {phased_times.mean():>9.2f}s')
print(f'{"momentum_sa_merged":<28} {merged_energies.mean():>12.4f} {merged_energies.std():>11.4f} {merged_times.mean():>9.2f}s')
print(f'{"Classical ground state":<28} {min_energy:>12.4f} {"-":>11} {"-":>10}')

methods = ['MomentumBuilder', 'Phased', 'Merged']
mean_e  = [mb_energies.mean(), phased_energies.mean(), merged_energies.mean()]
std_e   = [mb_energies.std(),  phased_energies.std(),  merged_energies.std()]
mean_t  = [mb_times.mean(),    phased_times.mean(),    merged_times.mean()]
std_t   = [mb_times.std(),     phased_times.std(),     merged_times.std()]
colors  = ['mediumseagreen', 'steelblue', 'salmon']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# -- Energy chart --
bars1 = ax1.bar(methods, mean_e, yerr=std_e, capsize=6,
                color=colors, edgecolor='black', linewidth=0.8)
ax1.axhline(min_energy, color='red', linestyle='--', linewidth=1.5,
            label=f'Classical ground state ({min_energy:.4f})')
for bar, m, s in zip(bars1, mean_e, std_e):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + s + 0.05,
             f'{m:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_ylabel('Energy', fontsize=12)
ax1.set_title(f'Final Energy (mean \u00b1 std, {NUM_TRIALS} trials)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_ylim(bottom=0)

# -- Runtime chart --
bars2 = ax2.bar(methods, mean_t, yerr=std_t, capsize=6,
                color=colors, edgecolor='black', linewidth=0.8)
for bar, m, s in zip(bars2, mean_t, std_t):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + s + 0.1,
             f'{m:.2f}s', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_ylabel('Wall Time (s)', fontsize=12)
ax2.set_title(f'Runtime (mean \u00b1 std, {NUM_TRIALS} trials)', fontsize=12, fontweight='bold')
ax2.set_ylim(bottom=0)

plt.suptitle('MomentumBuilder  vs  momentum_sa_phased  vs  momentum_sa_merged\n'
             f'(9-qubit Set Cover, iters={ITERS}, runs={OPT_RUNS})',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()